In [16]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [20]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import joblib  

mlflow.set_experiment("house_prices_models")

<Experiment: artifact_location='file:c:/Users/karen/Documents/proyecto_estadistica_multivariada/notebooks/mlruns/3', creation_time=1776070515897, experiment_id='3', last_update_time=1776070515897, lifecycle_stage='active', name='house_prices_models', tags={}, trace_location=None, workspace='default'>

In [17]:
X_train = pd.read_csv("../data/clean/X_train.csv")
y_train = pd.read_csv("../data/clean/y_train.csv").values.ravel()

In [21]:
preprocessor = joblib.load("../artifacts/preprocessing.pkl")

In [19]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scoring = {
    "rmse": make_scorer(rmse, greater_is_better=False),
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "r2": make_scorer(r2_score)
}

In [9]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso()
}

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models.update({
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(n_estimators=200),
    "XGBoost": XGBRegressor(n_estimators=200, verbosity=0),
    "LightGBM": LGBMRegressor(),
    "CatBoost": CatBoostRegressor(verbose=0)
})

In [11]:
from sklearn.neural_network import MLPRegressor

models["MLP"] = MLPRegressor(hidden_layer_sizes=(128,64), max_iter=500)

In [12]:
def train_model_cv(name, model, X, y, preprocessor, cv=5):
    
    from sklearn.pipeline import Pipeline
    
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    results = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    
    metrics = {
        "rmse_mean": -np.mean(results["test_rmse"]),
        "rmse_std": np.std(results["test_rmse"]),
        "mae_mean": -np.mean(results["test_mae"]),
        "r2_mean": np.mean(results["test_r2"])
    }
    
    return pipe, metrics

In [13]:
param_grids = {
    
    "Ridge": {
        "model__alpha": [0.1, 1.0, 10.0, 100.0]
    },
    
    "Lasso": {
        "model__alpha": [0.001, 0.01, 0.1, 1.0]
    },
    
    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5]
    },
    
    "XGBoost": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.01, 0.1],
        "model__max_depth": [3, 6]
    },
    
    "LightGBM": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.01, 0.1]
    },
    
    "CatBoost": {
        "model__depth": [4, 6, 8],
        "model__learning_rate": [0.01, 0.1]
    }
}

In [14]:
from sklearn.model_selection import GridSearchCV

def train_model(name, model, X, y, preprocessor, param_grids, cv=5):
    
    from sklearn.pipeline import Pipeline
    
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    if name in param_grids:
        
        grid = GridSearchCV(
            pipe,
            param_grid=param_grids[name],
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )
        
        grid.fit(X, y)
        
        best_model = grid.best_estimator_
        best_params = grid.best_params_
        best_score = -grid.best_score_
        
        return best_model, best_params, best_score
    
    else:
        # modelos sin tuning
        from sklearn.model_selection import cross_val_score
        
        scores = cross_val_score(
            pipe,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error"
        )
        
        return pipe, {}, -np.mean(scores)

In [ ]:
results_list = []

for name, model in models.items():
    
    with mlflow.start_run(run_name=name):
        
        best_model, best_params, rmse = train_model(
            name, model, X, y, preprocessor, param_grids
        )
        
        # log métricas
        mlflow.log_metric("rmse", rmse)
        
        # log parámetros
        for k, v in best_params.items():
            mlflow.log_param(k, v)
        
        # log modelo
        mlflow.sklearn.log_model(best_model, "model")
        
        results_list.append({
            "model": name,
            "rmse": rmse,
            **best_params
        })